# LSTM

In [43]:
import torch
import torch.nn as nn
from sklearn.model_selection import train_test_split
import pandas as pd
import numpy as np
from torch.utils.data import Dataset, DataLoader

In [161]:
data = "Artificial General Intelligence (AGI) is the hypothetical stage of artificial intelligence at which a machine possesses the broad, flexible, and adaptable cognitive abilities needed to understand, learn, reason, plan, solve problems, and apply knowledge across virtually any intellectual domain at a level comparable to, or potentially exceeding, humans, rather than being restricted to a narrow set of tasks for which it was specifically trained; unlike today's AI systems, which can often perform impressively on language, coding, image generation, or data analysis but still rely heavily on patterns learned from vast datasets and often struggle with robust reasoning, long-term autonomous planning, continual learning, real-world understanding, and reliable transfer of knowledge to entirely new situations, an AGI would be expected to acquire new skills with minimal instruction, combine concepts from different fields, adapt to unfamiliar environments, understand context deeply, learn from experience over time, maintain coherent goals, and function effectively across a wide range of activities without requiring task-specific redesign or retraining; for example, if presented with a completely new profession, scientific problem, business challenge, or creative endeavor, an AGI could study the relevant information, develop expertise, formulate strategies, execute plans, evaluate results, and improve its performance much like a capable human would, while also transferring insights from other domains to accelerate learning; however, there is no universally accepted definition of AGI, with some researchers defining it as human-level intelligence across most cognitive tasks, others viewing it as the ability to perform the majority of economically valuable work, and still others emphasizing generalization, autonomy, and adaptability rather than strict human equivalence, which is why debates frequently arise about whether increasingly capable AI models are approaching AGI or merely becoming more powerful examples of narrow AI; furthermore, AGI remains a theoretical goal rather than an achieved milestone because current systems still exhibit important limitations such as hallucinations, inconsistent reasoning, weak long-term memory, limited self-directed learning, poor awareness of real-world time and events unless explicitly informed, and difficulty maintaining reliable performance over extended autonomous tasks, and while some experts believe AGI could emerge through continued scaling of existing architectures combined with better reasoning, memory, and agentic capabilities, others argue that fundamentally new scientific breakthroughs may be required before machines can truly match the general, flexible, and continuously adaptive intelligence that humans display across the enormous variety of situations encountered in everyday life."

In [162]:
data = data.lower()
data = data.replace(",", " ")
data = data.replace(".", " ")
data = data.split()

In [163]:
vocab = data.copy()

In [169]:
len(data)

388

In [166]:
vocab = list(set(vocab))

In [168]:
vocab

['like',
 'possesses',
 'reliable',
 'perform',
 'life',
 'real-world',
 'if',
 'memory',
 'display',
 'virtually',
 'insights',
 'transfer',
 'creative',
 'capabilities',
 'deeply',
 'can',
 'examples',
 'requiring',
 'restricted',
 'combined',
 'furthermore',
 'everyday',
 'human',
 'about',
 'over',
 'than',
 'poor',
 'but',
 'adaptable',
 'strategies',
 'why',
 'combine',
 'completely',
 'and',
 'adaptability',
 'weak',
 'explicitly',
 'was',
 'image',
 'environments',
 'reasoning',
 'whether',
 'specifically',
 'increasingly',
 'are',
 'maintaining',
 'fundamentally',
 'an',
 'problem',
 'variety',
 'wide',
 'execute',
 'domains',
 'argue',
 'expertise',
 'most',
 'potentially',
 'merely',
 'maintain',
 'understand',
 'patterns',
 'accepted',
 'fields',
 'hypothetical',
 'set',
 'economically',
 'information',
 'more',
 'rely',
 'heavily',
 'profession',
 'new',
 'coding',
 'ai;',
 'relevant',
 'learning',
 'breakthroughs',
 'events',
 'entirely',
 'understanding',
 'instruction',

In [171]:
len(vocab)

264

In [172]:
vocabulary = {'<PAD>' : 0, '<UNK>' : 1, '<SOS>' : 2, '<EOS>' : 3}

In [173]:
toks = [i for i in range(len(vocabulary), len(vocab) + 1)]
deect = dict(zip(vocab, toks))
vocab = vocabulary | deect
vocab

{'<PAD>': 0,
 '<UNK>': 1,
 '<SOS>': 2,
 '<EOS>': 3,
 'like': 4,
 'possesses': 5,
 'reliable': 6,
 'perform': 7,
 'life': 8,
 'real-world': 9,
 'if': 10,
 'memory': 11,
 'display': 12,
 'virtually': 13,
 'insights': 14,
 'transfer': 15,
 'creative': 16,
 'capabilities': 17,
 'deeply': 18,
 'can': 19,
 'examples': 20,
 'requiring': 21,
 'restricted': 22,
 'combined': 23,
 'furthermore': 24,
 'everyday': 25,
 'human': 26,
 'about': 27,
 'over': 28,
 'than': 29,
 'poor': 30,
 'but': 31,
 'adaptable': 32,
 'strategies': 33,
 'why': 34,
 'combine': 35,
 'completely': 36,
 'and': 37,
 'adaptability': 38,
 'weak': 39,
 'explicitly': 40,
 'was': 41,
 'image': 42,
 'environments': 43,
 'reasoning': 44,
 'whether': 45,
 'specifically': 46,
 'increasingly': 47,
 'are': 48,
 'maintaining': 49,
 'fundamentally': 50,
 'an': 51,
 'problem': 52,
 'variety': 53,
 'wide': 54,
 'execute': 55,
 'domains': 56,
 'argue': 57,
 'expertise': 58,
 'most': 59,
 'potentially': 60,
 'merely': 61,
 'maintain': 62,
 

In [156]:
def word2tok(text):
    return vocab.get(text, vocab['<UNK>'])

In [174]:
for i in range(len(data)):
    data[i] = word2tok(data[i])

In [175]:
data

[127,
 242,
 246,
 115,
 159,
 103,
 67,
 131,
 193,
 127,
 246,
 177,
 158,
 146,
 250,
 5,
 103,
 138,
 132,
 37,
 32,
 91,
 228,
 196,
 168,
 63,
 154,
 187,
 233,
 216,
 129,
 37,
 247,
 174,
 111,
 13,
 122,
 198,
 264,
 177,
 146,
 155,
 243,
 168,
 197,
 60,
 133,
 163,
 106,
 29,
 200,
 22,
 168,
 146,
 221,
 68,
 193,
 220,
 255,
 158,
 227,
 41,
 46,
 92,
 205,
 224,
 234,
 120,
 158,
 19,
 116,
 7,
 114,
 210,
 175,
 76,
 42,
 143,
 197,
 202,
 256,
 31,
 176,
 72,
 73,
 210,
 64,
 160,
 117,
 150,
 89,
 37,
 116,
 134,
 207,
 211,
 44,
 104,
 98,
 152,
 113,
 79,
 9,
 83,
 37,
 6,
 15,
 193,
 174,
 168,
 82,
 75,
 251,
 51,
 102,
 90,
 157,
 165,
 168,
 124,
 75,
 166,
 207,
 93,
 84,
 35,
 164,
 117,
 171,
 66,
 217,
 168,
 170,
 43,
 63,
 204,
 18,
 154,
 117,
 130,
 28,
 208,
 62,
 119,
 101,
 37,
 180,
 213,
 111,
 146,
 54,
 260,
 193,
 99,
 97,
 21,
 167,
 145,
 197,
 96,
 255,
 206,
 10,
 212,
 207,
 146,
 36,
 75,
 74,
 252,
 52,
 192,
 178,
 197,
 16,
 215,
 51,
 1

In [199]:
class datalao(Dataset):
    def __init__(self, data, seq_len):
        self.data = data
        self.seq_len = seq_len
        
    def __len__(self):
        return len(self.data) - self.seq_len

    def __getitem__(self, idx):
        return(torch.tensor(self.data[idx : idx + self.seq_len]), torch.tensor(self.data[idx + 1 : idx + self.seq_len + 1]))

In [200]:
train_data = datalao(data, 4)

In [201]:
train_load = DataLoader(train_data, batch_size = 3)

In [202]:
class model(nn.Module):
    def __init__(self, data, vocab_size, hid_dim, embed_dim):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.lstm = nn.LSTM(embed_dim, hid_dim, batch_first = True)
        self.classy = nn.Linear(hid_dim, vocab_size)

    def forward(self, x):
        emb = self.embedding(x)
        output, (hidden, cell) = self.lstm(emb)
        logits = self.classy(output)
        return logits    

In [240]:
lstm = model(data, len(data), 128, 256)    

In [241]:
lr = 0.001
epochs = 100

In [242]:
crit = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(lstm.parameters(), lr = lr)

In [243]:
for epoch in range(epochs):
    tot_loss = 0
    for x, y  in train_load:
        logits = lstm(x)
        loss = crit(logits.reshape(-1, logits.shape[-1]), y.reshape(-1))
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        tot_loss += loss.item()
    avg_loss = tot_loss / len(train_load)
    print(f'epoch: {epoch} loss: {avg_loss}')

epoch: 0 loss: 5.935702931135893
epoch: 1 loss: 5.012437906116247
epoch: 2 loss: 3.54226890578866
epoch: 3 loss: 2.1562402425333858
epoch: 4 loss: 1.2628359436057508
epoch: 5 loss: 0.8134953312110156
epoch: 6 loss: 0.584977466147393
epoch: 7 loss: 0.456956482375972
epoch: 8 loss: 0.3823471059440635
epoch: 9 loss: 0.3367786420858465
epoch: 10 loss: 0.3070563087821938
epoch: 11 loss: 0.28659540179069154
epoch: 12 loss: 0.271878243831452
epoch: 13 loss: 0.2608662331767846
epoch: 14 loss: 0.2523397482873406
epoch: 15 loss: 0.24560661979194265
epoch: 16 loss: 0.24014393019024283
epoch: 17 loss: 0.23565493225760292
epoch: 18 loss: 0.23185706552612828
epoch: 19 loss: 0.22868524237856036
epoch: 20 loss: 0.22585522747976938
epoch: 21 loss: 0.22350976162124425
epoch: 22 loss: 0.22129718022915768
epoch: 23 loss: 0.21948195232835133
epoch: 24 loss: 0.2177649492296041
epoch: 25 loss: 0.2163201974290132
epoch: 26 loss: 0.21493661051863455
epoch: 27 loss: 0.21374276091592037
epoch: 28 loss: 0.2125913

In [209]:
torch.save(lstm.state_dict(), 'lstm_nwp.pth')

In [247]:
idx_to_word = {v:k for k,v in vocab.items()}

In [262]:
def generate(model, seed_tokens, idx_to_word, seq_len, max_new_tokens = 20):
    model.eval()
    generated = seed_tokens.copy()
    with torch.no_grad():
        for _ in range(max_new_tokens):
            context = generated[-seq_len:]
            x = torch.tensor([context], dtype=torch.long)
            logits = model(x)
            next_token = (logits[:, -1, :].argmax(dim = -1).item())
            generated.append(next_token)
    return " ".join(
        idx_to_word[t]
        for t in generated
    )

In [263]:
generate(lstm, data[0:2], idx_to_word, 5)

'artificial general intelligence (agi) is the hypothetical stage of artificial intelligence at which a machine possesses the broad flexible and adaptable cognitive'